In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "heart"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Get sheet names

In [3]:
metadata = pd.read_json("../metadata.json")
sheet_names = [obj["sheet_name"] for obj in metadata["tables"]]

sheet_names

['Table IV MarkerGene',
 'Table V SubclusterMarkerGene',
 'Table VI ChamberDiff',
 'Table VII SexDiff']

## Read in file

In [22]:
first_excel = pd.read_excel(table_name, sheet_name = sheet_names[:2], skiprows = 0)
excel = pd.read_excel(table_name, sheet_name = sheet_names[2:], skiprows = 1)

for e in list(first_excel.values()):
    print(e.columns)
for e in list(excel.values()):
    print(e.columns)

Index(['Cell Type', 'Gene', 'Ensembl ID', 'Chromosome', 'Pct.Target',
       'Pct.Other', 'avg_logFC', 'AUC', 'PPV50', 'Marker'],
      dtype='object')
Index(['Cell Type', 'Sub-cluster', 'Gene', 'Ensembl ID', 'Chromosome',
       'Pct.Target', 'Pct.Other', 'avg_logFC', 'AUC', 'PPV50', 'Marker'],
      dtype='object')
Index(['Gene', 'Ensembl ID', 'Chromosome', 'Cell Type', 'Pct.Atrium',
       'Pct.Ventricle', 'Effect', 'Effect SE', 'P', 'P.FDR.Adj', 'Pct.LA',
       'Pct.LV', 'Effect.1', 'Effect SE.1', 'P.1', 'P.FDR.Adj.1', 'Pct.RA',
       'Pct.RV', 'Effect.2', 'Effect SE.2', 'P.2', 'P.FDR.Adj.2', 'Pct.Left',
       'Pct.Right', 'Effect.3', 'Effect SE.3', 'P.3', 'P.FDR.Adj.3',
       'Pct.LA.1', 'Pct.RA.1', 'Effect.4', 'Effect SE.4', 'P.4', 'P.FDR.Adj.4',
       'Pct.LV.1', 'Pct.RV.1', 'Effect.5', 'Effect SE.5', 'P.5',
       'P.FDR.Adj.5'],
      dtype='object')
Index(['Gene', 'Ensembl ID', 'Chromosome', 'Cell Type', 'Pct.Male',
       'Pct.Female', 'Effect', 'Effect SE', 'P', 'P.FDR

In [23]:
cols = [
    {
        "sheet_name": sheet_names[0],
        "cols": {
            "avg_logFC": "logfc",
            "Cell Type": "cell_type_label",
            "Gene": "feature_name"
        }
    },
    {
        "sheet_name": sheet_names[1],
        "cols": {
            "avg_logFC": "logfc",
            "Cell Type": "cell_type_label",
            "Gene": "feature_name"
        }
    },
    {
        "sheet_name": sheet_names[2],
        "cols": {
            "P": "p_val",
            "P.FDR.Adj": "p_corr",
            "Cell Type": "cell_type_label",
            "Gene": "feature_name"
        }
    },
    {
        "sheet_name": sheet_names[3],
        "cols": {
            "P": "p_val",
            "P.FDR.Adj": "p_corr",
            "Cell Type": "cell_type_label",
            "Gene": "feature_name"
        }
    },
]

In [24]:
for idx, sheet in enumerate(cols):
    sname = sheet["sheet_name"]
    print(sname)
    excel_name = excel
    if idx < 2:
        excel_name = first_excel
    # only keep the columns we want
    excel_name[sname] = excel_name[sname].loc[:,sheet["cols"].keys()].rename(columns=sheet["cols"])
    # add sheet name to the dataframe
    excel_name[sname]["sheet_name"] = sname

Table IV MarkerGene
Table V SubclusterMarkerGene
Table VI ChamberDiff
Table VII SexDiff


In [37]:
first_df = pd.concat(list(first_excel.values()))
first_df["p_val"] = np.nan
first_df["p_corr"] = np.nan
second_df = pd.concat(list(excel.values()))
second_df["logfc"] = np.nan

In [38]:
first_df.head()

,logfc,cell_type_label,feature_name,sheet_name,p_val,p_corr
0,1.267279,1. Fibroblast I,DCN,Table IV MarkerGene,NaN,NaN
1,0.851194,1. Fibroblast I,LAMA2,Table IV MarkerGene,NaN,NaN
2,1.251984,1. Fibroblast I,NEGR1,Table IV MarkerGene,NaN,NaN
3,1.473957,1. Fibroblast I,ACSM3,Table IV MarkerGene,NaN,NaN
4,1.404422,1. Fibroblast I,ABCA6,Table IV MarkerGene,NaN,NaN


In [39]:
second_df.head()

,p_val,p_corr,cell_type_label,feature_name,sheet_name,logfc
0,2.020000e-32,3.00e-28*,Cardiomyocyte,NLGN1,Table VI ChamberDiff,NaN
1,1.220000e-30,9.08e-27*,Cardiomyocyte,NPPA,Table VI ChamberDiff,NaN
2,1.310000e-29,6.53e-26*,Cardiomyocyte,RP11-400K9.4,Table VI ChamberDiff,NaN
3,2.940000e-29,1.03e-25*,Cardiomyocyte,MYL4,Table VI ChamberDiff,NaN
4,3.450000e-29,1.03e-25*,Cardiomyocyte,CFAP61,Table VI ChamberDiff,NaN


In [42]:
df = pd.concat([first_df, second_df])

df.tail()

,logfc,cell_type_label,feature_name,sheet_name,p_val,p_corr
28,NaN,Fibroblast,PRPS1,Table VII SexDiff,1.000000e-02,0.176
29,NaN,Macrophage,XIST,Table VII SexDiff,2.910000e-09,2.14e-05*
30,NaN,Macrophage,RPS4Y1,Table VII SexDiff,8.690000e-07,0.001*
31,NaN,Macrophage,LINC00278,Table VII SexDiff,1.190000e-06,0.002*
32,NaN,NaN,Key: Pct: Percent of cells expressing gene at ...,Table VII SexDiff,NaN,NaN


## Unfiltered

In [43]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["logfc"],
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': '1. Fibroblast I',
   'cell_source': 'heart',
   'cell_state': None,
   'feature_name': 'DCN',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': '1. Fibroblast I',
   'cell_source': 'heart',
   'cell_state': None,
   'feature_name': 'DCN',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'Table IV MarkerGene',
   'source_metrics': {'p_corr': nan, 'log_fc': 1.26727921469976}}}]

## Save unfiltered

In [44]:
with open("evidence_unfiltered_metrics.json", 'w') as f:
    json.dump(result, f, indent=4)

## Perform the filtering

In [6]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "Pct.Target"

In [7]:
min_mean = 10
max_pval = 0.05
min_lfc = 1
max_gene_shares = 10
max_per_celltype = 20

# filter by pval and lfc
# dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")
dfc = df.query(f"Marker == 1.0 & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [8]:
markers[col_ct].value_counts()

celltype
9.Endothelium I                  20
Macrophage                       20
Vascular Smooth Muscle           20
Fibroblast III                   20
Endothelium II                   20
Ventricular Cardiomyocyte II     20
Ventricular Cardiomyocyte III    20
Adipocyte                        20
Atrial Cardiomyocyte             20
Ventricular Cardiomyocyte I      20
Pericyte                         18
Neuronal                         12
Fibroblast II                    12
Lymphocyte                        8
Fibroblast I                      8
Name: count, dtype: int64

In [9]:
if col_rank:
    print(markers.groupby(col_ct)[col_rank].mean().sort_values())

celltype
Lymphocyte                       0.564375
Neuronal                         0.617083
Pericyte                         0.656611
9.Endothelium I                  0.676100
Macrophage                       0.684300
Vascular Smooth Muscle           0.713900
Fibroblast III                   0.742450
Endothelium II                   0.753350
Fibroblast I                     0.807875
Fibroblast II                    0.821833
Ventricular Cardiomyocyte II     0.865550
Adipocyte                        0.872800
Ventricular Cardiomyocyte III    0.889950
Atrial Cardiomyocyte             0.932250
Ventricular Cardiomyocyte I      0.980800
Name: Pct.Target, dtype: float64


In [10]:
markers[col_ct].value_counts()

celltype
9.Endothelium I                  20
Macrophage                       20
Vascular Smooth Muscle           20
Fibroblast III                   20
Endothelium II                   20
Ventricular Cardiomyocyte II     20
Ventricular Cardiomyocyte III    20
Adipocyte                        20
Atrial Cardiomyocyte             20
Ventricular Cardiomyocyte I      20
Pericyte                         18
Neuronal                         12
Fibroblast II                    12
Lymphocyte                        8
Fibroblast I                      8
Name: count, dtype: int64

## Find Ensembl ID


In [11]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [12]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [13]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json

In [14]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [15]:
save

[{'cell_type_label': 'Lymphocyte',
  'gene': 'IKZF1',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000185811'},
 {'cell_type_label': 'Neuronal',
  'gene': 'GPM6B',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000046653'},
 {'cell_type_label': 'Neuronal',
  'gene': 'SHISA9',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000237515'},
 {'cell_type_label': 'Neuronal',
  'gene': 'SLC35F1',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000196376'},
 {'cell_type_label': '9.Endothelium I',
  'gene': 'PECAM1',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000261371'},
 {'cell_type_label': 'Pericyte',
  'gene': 'CARMN',
  'organism': 'homo_sapiens',
  'cell_source': 'heart',
  'cell_state': None,
  'gene_id': 'ENSG00000249669'},
 {'cell_t

## Save evidence

In [16]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 